# Imports and preprocessing

In [1]:
import csv
import numpy as np
from pathlib import Path
import sys, os
import json
import importlib
sys.path.append(os.path.join(os.getcwd(), '..'))

from utils import eda_helpers, all_functions, cv_andrea_dt, sampling_methods
import implementations

importlib.reload(eda_helpers)
importlib.reload(all_functions)
importlib.reload(cv_andrea_dt)
importlib.reload(sampling_methods)
importlib.reload(implementations)

from utils.eda_helpers import clean_data, read_csv_numeric
from utils.all_functions import sigmoid
from implementations import (
    logistic_regression,
    reg_logistic_regression,
    weighted_logistic_regression,
)

file_path = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
DATA_FOLDER = os.path.join(file_path, '..', 'dataset')

In [2]:
x_indices, x_data = read_csv_numeric(os.path.join(DATA_FOLDER, "x_train.csv"))
y_indices, y_data = read_csv_numeric(os.path.join(DATA_FOLDER, "y_train.csv"))

In [3]:
final_feature_set = ['_CASTHM1', 'FLUSHOT6', 'METVL11_', '_PASTRNG', 'EDUCA', 'CHCCOPD1', 'DRADVISE', 
                     'CHCKIDNY', 'ARTHSOCL', 'MENTHLTH', 'SMOKE100', 'DOCTDIAB', 'DIABAGE2', 'VETERAN3', 
                     'CHKHEMO3', 'SEX', '_DRNKWEK', 'JOINPAIN', 'MARITAL', 'EXERANY2', '_PACAT1', 'INSULIN', 
                     'DIFFALON', 'FRUIT1', 'QLACTLM2', 'BLIND', 'PADUR2_', 'PAFREQ2_', 'LMTJOIN3', 'CHOLCHK', 
                     'POORHLTH', 'DIABETE3', '_FRUTSUM', 'PERSDOC2', 'EXRACT21', 'FEETCHK', '_VEGESUM', 'PNEUVAC3', 
                     'BPHIGH4', 'GENHLTH', 'EXRACT11', '_BMI5', 'DECIDE', '_ASTHMS1', '_MRACE1', '_LMTSCL1', 'ORNGDAY_', 
                     'MSCODE', 'PHYSHLTH', 'TOLDHI2', 'PADUR1_', 'DIFFWALK', 'BEANDAY_', 'CHCSCNCR', 'PAFREQ1_', '_AIDTST3', 
                     'FLSHTMY2', 'PA1VIGM_', 'METVL21_', 'SXORIENT', 'ARTHDIS2', 'PA1MIN_', 'MAXVO2_', '_RACE_G1', 'LONGWTCH', 
                     'SCNTWRK1', 'FEETCHK2', 'EMPLOY1', 'BLOODCHO', 'USENOW3', 'DRNKANY5', '_PAREC1', 'DIFFDRES', 'USEEQUIP', 
                     'ADDEPEV2', '_AGE80', 'HTM4', 'INCOME2', 'CVDSTRK3', 'CHCOCNCR', 'MEDCOST', 'PREDIAB1', 'BPMEDS', 'HAVARTH3', 
                     'CHECKUP1', 'TRNSGNDR', 'CHILDREN']

In [4]:
del x_indices["Id"]
x_data = clean_data(x_data, x_indices)

In [5]:
# --- Keep only the columns of interest ---
x_data_interested = x_data[:, [x_indices[col] for col in final_feature_set]]
x_indices_interested = {col: i for i, col in enumerate(final_feature_set)}

# --- Drop columns that are all NaN ---
valid_cols = ~np.isnan(x_data_interested).all(axis=0)
x_data_interested = x_data_interested[:, valid_cols]
interesting_columns = [col for col, keep in zip(final_feature_set, valid_cols) if keep]
x_indices_interested = {col: i for i, col in enumerate(final_feature_set)}

# --- Impute remaining NaNs with column mean ---
col_mean = np.nanmean(x_data_interested, axis=0)
inds = np.where(np.isnan(x_data_interested))
x_data_interested[inds] = np.take(col_mean, inds[1])

# --- Standardize once ---
x_data_interested = (x_data_interested - np.mean(x_data_interested, axis=0)) / np.std(x_data_interested, axis=0)

# --- Prepare y ---
y = y_data[:,1].copy()
y[np.isnan(y)] = np.nanmean(y)
y = (y + 1) / 2  # convert -1,1 → 0,1
y = y.flatten()

# --- Checks ---
print("Unique labels:", np.unique(y))
print("Any NaN in features?", np.any(np.isnan(x_data_interested)))
print("Min std across features:", np.min(np.std(x_data_interested, axis=0)))
print("Label distribution (mean):", np.mean(y))

Unique labels: [0. 1.]
Any NaN in features? False
Min std across features: 0.9999999999999998
Label distribution (mean): 0.08830207079403295


# Evaluation

In [6]:
from utils.cv_andrea_dt import (
    cross_validate,
    train_logreg,
    predict,
    compute_training_error
)
from utils.sampling_methods import (
    undersample,
    oversample,
    smote
)


In [7]:
sampling_methods = {
    "none": None,
    "undersample": undersample,
    "oversample": oversample,
    "smote": smote
}
sampling_kwargs = {
    "undersample": {"ratio": 2.0, "seed": 42},
    "oversample": {"ratio": 3.0, "seed": 42},
    "smote": {"k": 5, "ratio": 3.0, "seed": 42, "batch_size": 20},
}

# all_results = []

In [8]:
from utils.cv_andrea_dt import analyze_cv_results, cross_validate_with_weights

In [16]:
all_results = []

for kind in ["w_logistic"]:
    print(f"\n=== MODEL: {kind.upper()} ===")

    for s_name, s_func in {"none": None}.items():
        print(f"\n--- Sampling: {s_name.upper()} ---")

        # prepare storage for fold-wise data
        weights_list = []
        train_metrics = []
        val_metrics = []
        class_weights_list = []

        # manual k-fold loop (same logic as inside cross_validate)
        n = len(y)
        rng = np.random.default_rng(42)
        indices = rng.permutation(n)
        folds = np.array_split(indices, 5)

        for i, val_idx in enumerate(folds):
            train_idx = np.setdiff1d(indices, val_idx)
            X_train, y_train = x_data_interested[train_idx], y[train_idx]
            X_val, y_val = x_data_interested[val_idx], y[val_idx]

            # optional resampling
            if s_func is not None:
                try:
                    X_train, y_train = s_func(X_train, y_train, **(sampling_kwargs.get(s_name, {}) or {}))
                except Exception as e:
                    print(f"[Warning] Sampling failed in fold {i+1}: {e}")
                    continue

            # train model
            w, loss = train_logreg(
                y_train, X_train,
                kind=kind,
                gamma=0.4,
                lambda_=0.01,
                max_iters=500,
                verbose=False
            )
            weights_list.append(w)

            # (optional) if class weights exist in model call, store them
            # class_weights_list.append({"0": w0, "1": w1})  # if applicable
            # compute metrics on train + val
            train_m = compute_training_error(y_train, X_train, w, threshold=0.8, returning="all")
            val_m   = compute_training_error(y_val, X_val, w, threshold=0.8, returning="all")
            train_metrics.append(train_m)
            val_metrics.append(val_m)

            print(f"Fold {i+1}/5 → "
                  f"Train F1={train_m['f1']:.3f}, Val F1={val_m['f1']:.3f}, "
                  f"AUC={val_m['roc_auc']:.3f}, Loss={loss:.4f}")
            w = w.ravel()  # or w.flatten(), shape becomes (n_features,)
            for name, val in zip(interesting_columns, w):
                print(f"  {name:20s}: {val:+.4f}")
            for name, val in zip(interesting_columns, w):
                print(f"  {name:20s}: {val:+.4f}")
    
    all_weights = np.array(weights_list)  # shape: (K, n_features)
    var_per_feature = np.var(all_weights, axis=0)

    print("\n=== Variance of weights across folds ===")
    for name, var in zip(interesting_columns, var_per_feature):
        print(f"{name:20s}: variance = {var:.6f}")



=== MODEL: W_LOGISTIC ===

--- Sampling: NONE ---
Fold 1/5 → Train F1=0.421, Val F1=0.422, AUC=0.144, Loss=0.5187
  _CASTHM1            : -0.0135
  FLUSHOT6            : +0.0073
  METVL11_            : -0.0045
  _PASTRNG            : -0.0037
  EDUCA               : -0.0247
  CHCCOPD1            : -0.1326
  DRADVISE            : -0.0637
  CHCKIDNY            : -0.0843
  ARTHSOCL            : +0.0062
  MENTHLTH            : +0.0041
  SMOKE100            : -0.0817
  DOCTDIAB            : -0.0005
  DIABAGE2            : -0.0055
  VETERAN3            : -0.0595
  CHKHEMO3            : +0.0073
  SEX                 : -0.2451
  _DRNKWEK            : -0.0008
  JOINPAIN            : +0.0224
  MARITAL             : -0.0073
  EXERANY2            : +0.0054
  _PACAT1             : -0.0105
  INSULIN             : -0.0260
  DIFFALON            : -0.0112
  FRUIT1              : +0.0111
  QLACTLM2            : -0.0449
  BLIND               : -0.0362
  PADUR2_             : +0.0055
  PAFREQ2_           

TypeError: unsupported format string passed to numpy.ndarray.__format__

In [17]:
# Convert weights_list to a proper 2D array
all_weights = np.array(weights_list)      # shape might be (num_folds, n_features, 1)
all_weights = np.squeeze(all_weights)     # remove any singleton dimensions → (num_folds, n_features)
all_weights = all_weights.reshape(len(weights_list), -1)  # ensure exactly 2D

# Compute variance across folds (axis=0)
var_per_feature = np.var(all_weights, axis=0)

# Print results
print("\n=== Variance of weights across folds ===")
for name, var in zip(interesting_columns, var_per_feature):
    print(f"{name:20s}: variance = {float(var):.6f}")  # cast to scalar



=== Variance of weights across folds ===
_CASTHM1            : variance = 0.000011
FLUSHOT6            : variance = 0.000005
METVL11_            : variance = 0.000005
_PASTRNG            : variance = 0.000007
EDUCA               : variance = 0.000002
CHCCOPD1            : variance = 0.000005
DRADVISE            : variance = 0.000005
CHCKIDNY            : variance = 0.000007
ARTHSOCL            : variance = 0.000002
MENTHLTH            : variance = 0.000004
SMOKE100            : variance = 0.000002
DOCTDIAB            : variance = 0.000005
DIABAGE2            : variance = 0.000001
VETERAN3            : variance = 0.000002
CHKHEMO3            : variance = 0.000005
SEX                 : variance = 0.000003
_DRNKWEK            : variance = 0.000003
JOINPAIN            : variance = 0.000005
MARITAL             : variance = 0.000005
EXERANY2            : variance = 0.000004
_PACAT1             : variance = 0.000016
INSULIN             : variance = 0.000002
DIFFALON            : variance = 0